# Unsupervised Learning

## Importing Essential Packages

In [6]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dot, Lambda

## K-Means Clustering

## Recommender Systems

### Collaborative Filtering Algorithm

In [ ]:
# Automatically finds the Derivative of the cost function
# Auto Diff (automatic Differentiation)

w = tf.Variable(3.0)
x = 1.0
y = 1.0
alpha = 0.01

iterations = 200

for iter in range(iterations):
    # used to compute the cost J to enable auto differentiation
    with tf.GradientTape() as tape:
        fwb = w * x
        costJ = (fwb - y) ** 2
    # used the gradient tape to calculate the gradients
    # of the cost with respect to the parameter w
    [dJdw] = tape.gradient(costJ, [w])

    # Run one step of gradient descent by updating
    # the value of w to reduce the cost
    w.assign_add(-alpha * dJdw)



In [7]:
w

<tf.Variable 'Variable:0' shape=() dtype=float32, numpy=1.0351759195327759>

In [9]:
num_movies = 4
num_users = 3
num_features = 2

Y = np.array([
    [5, 4, 0],
    [3, 0, 0],
    [4, 0, 0],
    [3, 0, 0]
], dtype=float)

R = (Y != 0).astype(float)

Ymean = np.sum(Y, axis=1) / (np.sum(R, axis=1) + 1e-8)
Ynorm = Y - Ymean.reshape(-1, 1)

Ynorm = tf.constant(Ynorm, dtype=tf.float32)
R = tf.constant(R, dtype=tf.float32)

X = tf.Variable(tf.random.normal((num_movies, num_features)))
W = tf.Variable(tf.random.normal((num_users, num_features)))
b = tf.Variable(tf.zeros((1, num_users)))

lambda_ = 0.1

def cofiCostFuncV(X, W, b, Y, R, num_users, num_movies, lambda_):
    predictions = tf.matmul(X, W, transpose_b=True) + b
    error = (predictions - Y) * R
    cost = 0.5 * tf.reduce_sum(tf.square(error))
    
    cost += (lambda_/2) * (
        tf.reduce_sum(tf.square(X)) + tf.reduce_sum(tf.square(W))
    )
    
    return cost

optimizer = keras.optimizers.Adam(learning_rate = 1e-1)

iterations = 200
for iter in range(iterations):

    with tf.GradientTape() as tape:
        cost_value = cofiCostFuncV(X, W, b, Ynorm, R, num_users, num_movies, lambda_)

        grads = tape.gradient(cost_value, [X, W, b])

        optimizer.apply_gradients(zip(grads, [X, W, b]))

### Content-based Filtering

In [9]:
user_NN = tf.keras.models.Sequential([
    Dense(256, activation = 'relu'),
    Dense(128, activation = 'relu'),
    Dense(32)
])

item_NN = tf.keras.models.Sequential([
    Dense(256, activation = 'relu'),
    Dense(128, activation = 'relu'),
    Dense(32)
])

num_user_features = 10
num_item_features = 8

N = 1000

user_features = np.random.rand(N, num_user_features).astype(np.float32)

item_features = np.random.rand(N, num_item_features).astype(np.float32)

labels = np.random.rand(N, 1).astype(np.float32)

input_user = Input(shape=(num_user_features,))
vu = user_NN(input_user)
vu = keras.ops.normalize(vu, axis=1)

input_item = Input(shape=(num_item_features,))
vm = item_NN(input_item)
vm = keras.ops.normalize(vm, axis=1)

output = Dot(axes = 1)([vu, vm])

model = Model([input_user, input_item], output)

cost_fn = tf.keras.losses.MeanSquaredError()